# Actividad 8. Formulación del problema y preparación de datos

Antes de entrenar cualquier modelo hay que responder tres preguntas:

1. **¿Qué se predice?** Si un evento será de alto impacto y cuántas personas
   y cuánto dinero podría costar.
2. **¿Con qué información?** Solo variables conocidas al inicio del evento.
3. **¿Cómo se evalúa sin engañarnos?** Con una partición temporal y sin usar
   el conjunto de prueba para ajustar nada.

Este notebook deja listo el dataset de modelado, documenta las variables y
comprueba que no haya fuga de información.

## 1. Carga y contexto

In [1]:

import json
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

MODELOS = RAIZ / "modelos"
RESULTADOS = RAIZ / "resultados" / "ml"
GRAFICAS = RAIZ / "resultados" / "graficas" / "ml"
TABLAS = RAIZ / "resultados" / "tablas"
for carpeta in (RESULTADOS, GRAFICAS, TABLAS):
    carpeta.mkdir(parents=True, exist_ok=True)

PALETA = {
    "purpura": "#4C1D95",
    "purpura_claro": "#8B5CF6",
    "rosa": "#FA8A9A",
    "amarillo": "#FFB71B",
    "rojo": "#EE3E29",
    "coral": "#FF8A5F",
    "azul": "#1B4576",
    "gris": "#7A7191",
}
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 150,
        "font.family": "DejaVu Sans",
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.22,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
    }
)


def guardar(fig, nombre):
    ruta = GRAFICAS / nombre
    fig.savefig(ruta, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Figura guardada: {ruta.relative_to(RAIZ)}")

from src.ml.datos import cargar_y_preparar, resumen_particiones

datos = cargar_y_preparar()
df = datos["df"]
train, val, test = datos["train"], datos["val"], datos["test"]

print(f"Eventos entre 2000 y 2025: {len(df):,}")
print(f"Eventos con algún indicador de impacto: {int(df['elegible_clasificacion'].sum()):,}")
print(f"Umbral de fallecimientos (P75 de entrenamiento): {datos['umbrales']['muertes_p75']:.0f}")
print(f"Umbral de personas afectadas (P75 de entrenamiento): {datos['umbrales']['afectados_p75']:,.0f}")
print()
print(datos["umbrales"]["definicion"])


Eventos entre 2000 y 2025: 10,741
Eventos con algún indicador de impacto: 10,173
Umbral de fallecimientos (P75 de entrenamiento): 35
Umbral de personas afectadas (P75 de entrenamiento): 50,000

Evento de alto impacto si los fallecimientos son mayores o iguales al percentil 75 de entrenamiento o si los afectados son mayores o iguales a su percentil 75.


## 2. Partición temporal

El orden importa: se entrena con el pasado y se evalúa con el futuro. Nunca
se mezclan años entre particiones.

* **Entrenamiento:** 2000–2022.
* **Validación:** 2023. Aquí se eligen hiperparámetros, umbral y calibración.
* **Prueba:** 2024–2025. Se toca una sola vez, al final.
* **2026:** se excluye porque el año está incompleto y no tiene PIB.

In [2]:
particiones = resumen_particiones(train, val, test)
display(particiones)
particiones.to_csv(TABLAS / "actividad8_particiones.csv", index=False)

fig, ejes = plt.subplots(1, 2, figsize=(11, 4.2))
colores = [PALETA["purpura"], PALETA["amarillo"], PALETA["rojo"]]
ejes[0].bar(particiones["particion"], particiones["elegibles_clasificacion"], color=colores)
for i, valor in enumerate(particiones["elegibles_clasificacion"]):
    ejes[0].text(i, valor + 80, f"{valor:,}", ha="center", fontsize=9)
ejes[0].set_title("Eventos utilizables por partición")
ejes[0].set_ylabel("Eventos")

ejes[1].bar(particiones["particion"], particiones["prevalencia_alto_impacto"], color=colores)
for i, valor in enumerate(particiones["prevalencia_alto_impacto"]):
    ejes[1].text(i, valor + 0.6, f"{valor:.1f} %", ha="center", fontsize=9)
ejes[1].set_title("Prevalencia de alto impacto")
ejes[1].set_ylabel("Porcentaje")
ejes[1].set_ylim(0, 48)
guardar(fig, "actividad8_particiones.png")


,particion,anios,eventos,elegibles_clasificacion,alto_impacto,prevalencia_alto_impacto,con_afectados,con_dano,con_magnitud
0,Entrenamiento,2000–2022,9420,8913,2943,33.02,7655,2846,3106
1,Validación,2023–2023,462,447,151,33.78,359,91,74
2,Prueba,2024–2025,859,813,321,39.48,677,193,159


Figura guardada: resultados\graficas\ml\actividad8_particiones.png


## 3. Variable objetivo

`alto_impacto = 1` cuando el evento alcanza **cualquiera** de los dos umbrales
del percentil 75 calculado **solo con entrenamiento**:

$$alto\_impacto = [\text{fallecidos} \ge 35] \lor [\text{afectados} \ge 50{,}000]$$

La clase positiva ronda un tercio de los eventos, así que no hay un desbalance
extremo. Aun así se usan pesos de clase y métricas como PR-AUC y F1, que son
más informativas que la exactitud cuando el positivo es minoritario.

In [3]:
elegibles = df[df["elegible_clasificacion"] == 1]
conteo = elegibles["alto_impacto"].value_counts().sort_index()
print("Clase 0 (impacto ordinario):", f"{conteo.get(0, 0):,}")
print("Clase 1 (alto impacto)     :", f"{conteo.get(1, 0):,}")
print(f"Prevalencia global: {elegibles['alto_impacto'].mean() * 100:.2f} %")

fig, ejes = plt.subplots(1, 2, figsize=(12, 4.2))
ejes[0].bar(["Impacto ordinario", "Alto impacto"], conteo.values, color=[PALETA["rosa"], PALETA["rojo"]])
ejes[0].set_title("Distribución de la variable objetivo")
ejes[0].set_ylabel("Eventos")

valores = elegibles["log_afectados"].dropna()
ejes[1].hist(valores, bins=45, color=PALETA["purpura_claro"], edgecolor="white")
ejes[1].axvline(
    np.log1p(datos["umbrales"]["afectados_p75"]),
    color=PALETA["rojo"],
    linestyle="--",
    linewidth=2,
    label="Umbral P75 de afectados",
)
ejes[1].set_title("Personas afectadas (escala logarítmica)")
ejes[1].set_xlabel("log1p(afectados)")
ejes[1].set_ylabel("Eventos")
ejes[1].legend(frameon=False)
guardar(fig, "actividad8_objetivo.png")


Clase 0 (impacto ordinario): 6,758
Clase 1 (alto impacto)     : 3,415
Prevalencia global: 33.57 %
Figura guardada: resultados\graficas\ml\actividad8_objetivo.png


## 4. Variables de entrada y control de fuga

La regla es tajante: **el modelo no puede ver el resultado que intenta
predecir**. Quedan prohibidos fallecidos, heridos, afectados, daños, ayuda,
declaratorias, costos de reconstrucción y fechas de actualización.

Sí se permiten variables que existen al inicio del evento: tipo y subtipo,
lugar, mes, magnitud, duración, contexto socioeconómico y el historial del
país en años anteriores.

In [4]:
from src.ml import config

variables = [
    ("tipo", "Categórica", "Disaster Type", "Se conoce al inicio del evento"),
    ("subtipo", "Categórica", "Disaster Subtype", "Se conoce al inicio del evento"),
    ("subgrupo", "Categórica", "Disaster Subgroup", "Se conoce al inicio del evento"),
    ("region", "Categórica", "Región EM-DAT", "Contexto geográfico"),
    ("subregion", "Categórica", "Subregión EM-DAT", "Contexto geográfico"),
    ("iso", "Categórica", "País", "Contexto geográfico"),
    ("escala", "Categórica", "Magnitude Scale", "Unidad de la magnitud"),
    ("pib_pc_log", "Numérica", "PIB / población (WDI)", "Indicador del año o anterior"),
    ("poblacion_log", "Numérica", "Población (WDI)", "Indicador del año o anterior"),
    ("densidad_log", "Numérica", "Densidad (WDI)", "Indicador del año o anterior"),
    ("mes_sin", "Numérica", "Mes de inicio", "Estacionalidad"),
    ("mes_cos", "Numérica", "Mes de inicio", "Estacionalidad"),
    ("duracion_log", "Numérica", "Fechas de inicio y fin", "Duración declarada"),
    ("magnitud_z", "Numérica", "Magnitude", "Magnitud estandarizada por escala"),
    ("magnitud_faltante", "Bandera", "Magnitude", "Indica si no se reportó"),
    ("mes_faltante", "Bandera", "Start Month", "Indica si no se reportó"),
    ("duracion_faltante", "Bandera", "Fechas", "Indica si no se reportó"),
    ("pib_faltante", "Bandera", "PIB", "Indica si no está disponible"),
]
tabla_variables = pd.DataFrame(variables, columns=["variable", "tipo", "origen", "justificación"])
display(tabla_variables)
tabla_variables.to_csv(TABLAS / "actividad8_variables.csv", index=False)

prohibidas = [
    "Total Deaths", "No. Injured", "Total Affected", "Total Damage",
    "Reconstruction", "Insured", "AID Contribution", "Appeal",
    "Declaration", "Entry Date", "Last Update", "CPI",
]
entradas = config.CAT_COLS + config.NUM_COLS_MODELO
fugas = [columna for columna in entradas if any(p.lower() in columna.lower() for p in prohibidas)]
assert fugas == [], f"fuga de información: {fugas}"
print("Control de fuga superado: ninguna entrada coincide con una variable de impacto.")


,variable,tipo,origen,justificación
0,tipo,Categórica,Disaster Type,Se conoce al inicio del evento
1,subtipo,Categórica,Disaster Subtype,Se conoce al inicio del evento
2,subgrupo,Categórica,Disaster Subgroup,Se conoce al inicio del evento
3,region,Categórica,Región EM-DAT,Contexto geográfico
4,subregion,Categórica,Subregión EM-DAT,Contexto geográfico
5,iso,Categórica,País,Contexto geográfico
6,escala,Categórica,Magnitude Scale,Unidad de la magnitud
7,pib_pc_log,Numérica,PIB / población (WDI),Indicador del año o anterior
8,poblacion_log,Numérica,Población (WDI),Indicador del año o anterior
9,densidad_log,Numérica,Densidad (WDI),Indicador del año o anterior


Control de fuga superado: ninguna entrada coincide con una variable de impacto.


## 5. Cobertura de los datos

La magnitud falta en dos tercios de los eventos y el daño económico en cerca
del 70 %. En lugar de borrar esos registros, se usan banderas de faltante y
modelos que toleran valores ausentes (XGBoost) o imputación por mediana
(red neuronal).

La regresión principal es **personas afectadas**; la de daño económico se
reporta como análisis exploratorio.

In [5]:
filas = []
for nombre, parte in [("Entrenamiento", train), ("Validación", val), ("Prueba", test)]:
    elegible = parte[parte["elegible_clasificacion"] == 1]
    total = len(elegible)
    filas.append(
        {
            "particion": nombre,
            "eventos": total,
            "con_magnitud": elegible["magnitud"].notna().mean(),
            "con_afectados": elegible["Total Affected"].notna().mean(),
            "con_dano": elegible["log_dano"].notna().mean(),
            "con_pib": (elegible["pib_faltante"] == 0).mean(),
        }
    )
cobertura = pd.DataFrame(filas)
display(cobertura.style.format({c: "{:.1%}" for c in cobertura.columns if c != "particion" and c != "eventos"}))
cobertura.to_csv(TABLAS / "actividad8_cobertura.csv", index=False)

fig, eje = plt.subplots(figsize=(8.5, 4.2))
campos = ["con_magnitud", "con_afectados", "con_dano", "con_pib"]
etiquetas = ["Magnitud", "Afectados", "Daño económico", "PIB"]
ancho = 0.2
for i, (_, fila) in enumerate(cobertura.iterrows()):
    eje.bar(
        np.arange(len(campos)) + (i - 1) * ancho,
        [fila[c] * 100 for c in campos],
        width=ancho,
        label=fila["particion"],
        color=[PALETA["purpura"], PALETA["amarillo"], PALETA["rojo"]][i],
    )
eje.set_xticks(np.arange(len(campos)))
eje.set_xticklabels(etiquetas)
eje.set_ylabel("Cobertura (%)")
eje.set_title("Disponibilidad de campos clave por partición")
eje.legend(frameon=False)
guardar(fig, "actividad8_cobertura.png")


,particion,eventos,con_magnitud,con_afectados,con_dano,con_pib
0,Entrenamiento,8913,34.8%,85.9%,31.9%,98.1%
1,Validación,447,16.6%,80.3%,20.4%,96.6%
2,Prueba,813,19.6%,83.3%,23.7%,95.0%


Figura guardada: resultados\graficas\ml\actividad8_cobertura.png


## 6. Guardado del dataset de modelado

Se conserva una copia auditable con las variables de entrada y los objetivos.
El archivo no se publica en la app: es evidencia para revisar y reproducir.

In [6]:
from src.ml.datos import guardar_dataset_modelado

ruta = guardar_dataset_modelado(df)
print(f"Dataset de modelado: {ruta.relative_to(RAIZ)}")
print(f"Filas: {len(df):,} · Columnas: {df.shape[1]}")


Dataset de modelado: resultados\ml\dataset_modelado.csv
Filas: 10,741 · Columnas: 98


## Conclusión

* 8,913 eventos de entrenamiento, 447 de validación y 813 de prueba.
* La clase de alto impacto representa 33.0 % en entrenamiento y 39.5 % en
  prueba. El cambio se reporta y se analiza por año en la Actividad 12.
* Los umbrales se calcularon solo con entrenamiento.
* Ninguna variable de impacto entra al modelo.

Con este dataset se puede entrenar sin fuga de información. La siguiente
actividad construye la red neuronal de clasificación.